In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [22]:
CSV_PATH = "../../data/raw_dataset/GLCM/unique_crop_window_glcm_radiomics_dataset.csv"
OUT_CSV  = "../data/processed_dataset/final_2d_cleaned_dataset_v2.csv"

In [23]:
dataset = pd.read_csv(CSV_PATH)

print("=== Shape ===")
print(dataset.shape)

print("\n=== Feature families present ===")
for prefix in ["stat_", "cm_", "rlm_", "glszm_", "ngtdm_"]:
    cols = [c for c in dataset.columns if c.startswith(prefix)]
    print(f"{prefix}: {len(cols)} features")

print("\n=== NaN analysis ===")
nan_rates = dataset.isnull().mean().sort_values(ascending=False)
print(f"Features with any NaN: {(nan_rates > 0).sum()}")
print(f"Features with >5% NaN: {(nan_rates > 0.05).sum()}")
print(f"Features with >20% NaN: {(nan_rates > 0.20).sum()}")
print(f"Features with >50% NaN: {(nan_rates > 0.50).sum()}")
print("\nTop 20 features by NaN rate:")
print(nan_rates.head(20))

print("\n=== Inf analysis ===")
inf_counts = dataset.replace([np.inf, -np.inf], np.nan).isnull().sum() - dataset.isnull().sum()
print(f"Features with any Inf: {(inf_counts > 0).sum()}")
print(inf_counts[inf_counts > 0])

print("\n=== Constant features (zero variance) ===")
numeric_cols = dataset.select_dtypes(include=[np.number]).columns
std_vals = dataset[numeric_cols].std()
constant_cols = std_vals[std_vals == 0].index.tolist()
print(f"Constant features: {len(constant_cols)}")
print(constant_cols)

print("\n=== Sample counts ===")
print(f"Total samples: {len(dataset)}")
if "patient_id" in dataset.columns:
    print(f"Unique patients: {dataset['patient_id'].nunique()}")
if "organ" in dataset.columns:
    print(f"Organs: {dataset['organ'].value_counts()}")
if "is_augmented" in dataset.columns:
    print(f"Real masks: {(dataset['is_augmented']==0).sum()}")
    print(f"Augmented masks: {(dataset['is_augmented']==1).sum()}")

print("\n=== Outlier summary for texture features ===")
texture_cols = [c for c in dataset.columns 
                if any(c.startswith(p) for p in ["cm_","glrlm_","glszm_","ngtdm_"])]
if texture_cols:
    for col in texture_cols[:5]:  # sample of first 5
        s = dataset[col].dropna()
        p1, p99 = s.quantile(0.01), s.quantile(0.99)
        outliers = ((s < p1) | (s > p99)).sum()
        print(f"{col}: min={s.min():.3f}, median={s.median():.3f}, "
              f"max={s.max():.3f}, outliers(p1/p99)={outliers}")

=== Shape ===
(50719, 57)

=== Feature families present ===
stat_: 0 features
cm_: 25 features
rlm_: 0 features
glszm_: 0 features
ngtdm_: 0 features

=== NaN analysis ===
Features with any NaN: 16
Features with >5% NaN: 15
Features with >20% NaN: 15
Features with >50% NaN: 15

Top 20 features by NaN rate:
image_file_name                   1.00000
image_series_description          1.00000
image_directory                   1.00000
image_study_date                  1.00000
image_study_description           1.00000
image_pet_suv_type                1.00000
image_series_instance_uid         1.00000
image_mask_directory              1.00000
image_mask_file_name              1.00000
image_settings_id                 1.00000
image_mask_randomise_id           1.00000
image_mask_series_description     1.00000
image_mask_series_instance_uid    1.00000
image_noise_iteration_id          1.00000
image_noise_level                 1.00000
cm_inv_var_d1_2d_avg_fbn_n16      0.00002
image_modality      

## cleaning

In [24]:
METADATA_COLS = [
    "sample_name", "image_modality", "is_augmented", "organ",
    "image_file_name", "image_directory", "image_study_date",
    "image_study_description", "image_series_instance_uid",
    "image_series_description", "image_mask_file_name",
    "image_pet_suv_type", "image_settings_id",
    "image_mask_series_instance_uid", "image_mask_directory",
    "image_mask_series_description", "image_noise_level",
    "image_noise_iteration_id", "image_mask_randomise_id",
    "image_voxel_size_x", "image_voxel_size_y", "image_voxel_size_z",
    "image_rotation_angle", "image_translation_x",
    "image_translation_y", "image_translation_z",
    "image_mask_adapt_size", "image_mask_label",
    "image_mask_randomise_id"
]

# Keep only columns that exist in the dataframe
meta_to_drop = [c for c in METADATA_COLS if c in dataset.columns]
dataset = dataset.drop(columns=meta_to_drop)
print(f"\nAfter dropping metadata: {dataset.shape}")


After dropping metadata: (50719, 29)


In [25]:
dataset = dataset.replace([np.inf, -np.inf], np.nan)
print(f"After replacing Inf: {dataset.shape}")

After replacing Inf: (50719, 29)


In [26]:
NON_FEATURE_COLS = [
    "patient_id", "organ", "is_augmented",
    "ct_image_path", "mask_path"
]
all_feature_cols = [c for c in dataset.columns if c not in NON_FEATURE_COLS]
feature_cols = dataset[all_feature_cols].select_dtypes(include=[np.number]).columns.tolist()

nan_rates = dataset[feature_cols].isnull().mean()
high_nan_cols = nan_rates[nan_rates > 0.20].index.tolist()
print(f"\nFeatures dropped (NaN rate > 20%): {len(high_nan_cols)}")
print(high_nan_cols)


Features dropped (NaN rate > 20%): 0
[]


In [27]:
dataset = dataset.drop(columns=high_nan_cols)
all_feature_cols = [c for c in dataset.columns if c not in NON_FEATURE_COLS]
feature_cols = dataset[all_feature_cols].select_dtypes(include=[np.number]).columns.tolist()
print(f"After dropping high-NaN features: {dataset.shape}")

After dropping high-NaN features: (50719, 29)


In [28]:
# std over numeric only (string columns excluded)
numeric_feature_cols = dataset[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
std_vals = dataset[numeric_feature_cols].std()
constant_cols = std_vals[std_vals < 1e-6].index.tolist()
print(f"\nConstant/near-constant features dropped: {len(constant_cols)}")
print(constant_cols)


dataset = dataset.drop(columns=constant_cols)
all_feature_cols = [c for c in dataset.columns if c not in NON_FEATURE_COLS]
feature_cols = dataset[all_feature_cols].select_dtypes(include=[np.number]).columns.tolist()
print(f"After dropping constant features: {dataset.shape}")


Constant/near-constant features dropped: 0
[]
After dropping constant features: (50719, 29)


In [29]:
before = len(dataset)
dataset = dataset.dropna(subset=feature_cols)
after = len(dataset)
print(f"\nRows dropped (NaN in features): {before - after} "
      f"({100*(before-after)/before:.1f}%)")
print(f"After dropping NaN rows: {dataset.shape}")


Rows dropped (NaN in features): 1 (0.0%)
After dropping NaN rows: (50718, 29)


In [30]:
print("\n=== Final Dataset Summary ===")
print(f"Total samples  : {len(dataset)}")
print(f"Feature columns: {len(feature_cols)}")
print(f"Unique patients: {dataset['patient_id'].nunique()}")

print("\n=== Feature column names ===")
print(feature_cols)


=== Final Dataset Summary ===
Total samples  : 50718
Feature columns: 26
Unique patients: 6900

=== Feature column names ===
['cm_joint_max_d1_2d_avg_fbn_n16', 'cm_joint_avg_d1_2d_avg_fbn_n16', 'cm_joint_var_d1_2d_avg_fbn_n16', 'cm_joint_entr_d1_2d_avg_fbn_n16', 'cm_diff_avg_d1_2d_avg_fbn_n16', 'cm_diff_var_d1_2d_avg_fbn_n16', 'cm_diff_entr_d1_2d_avg_fbn_n16', 'cm_sum_avg_d1_2d_avg_fbn_n16', 'cm_sum_var_d1_2d_avg_fbn_n16', 'cm_sum_entr_d1_2d_avg_fbn_n16', 'cm_energy_d1_2d_avg_fbn_n16', 'cm_contrast_d1_2d_avg_fbn_n16', 'cm_dissimilarity_d1_2d_avg_fbn_n16', 'cm_inv_diff_d1_2d_avg_fbn_n16', 'cm_inv_diff_norm_d1_2d_avg_fbn_n16', 'cm_inv_diff_mom_d1_2d_avg_fbn_n16', 'cm_inv_diff_mom_norm_d1_2d_avg_fbn_n16', 'cm_inv_var_d1_2d_avg_fbn_n16', 'cm_corr_d1_2d_avg_fbn_n16', 'cm_auto_corr_d1_2d_avg_fbn_n16', 'cm_clust_tend_d1_2d_avg_fbn_n16', 'cm_clust_shade_d1_2d_avg_fbn_n16', 'cm_clust_prom_d1_2d_avg_fbn_n16', 'cm_info_corr1_d1_2d_avg_fbn_n16', 'cm_info_corr2_d1_2d_avg_fbn_n16', 'z_middle_global

In [32]:
dataset.to_csv(
    "../../data/processed_dataset/GLCM/unique_crop_window_glcm_radiomics_dataset_cleaned.csv",
    index=False
)
print(f"\nSaved cleaned dataset: {len(dataset)} samples, "
      f"{len(feature_cols)} features")


Saved cleaned dataset: 50718 samples, 26 features
